## 2. OSM Overpass API: Estructura Urbana

Consultaremos la API de Overpass para obtener elementos de infraestructura urbana.
Ejemplo: Comisarías de policía (`amenity=police`) en un área específica (ej. Madrid).

In [7]:
import requests
import folium
import time
import os
import pandas as pd

overpass_urls = [
    "https://overpass-api.de/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter"
]

overpass_query = """
[out:json][timeout:90];
(
  node["amenity"="police"](40.49,-74.26,40.92,-73.69);
  way["amenity"="police"](40.49,-74.26,40.92,-73.69);
  relation["amenity"="police"](40.49,-74.26,40.92,-73.69);
);
out center;
"""

data = None

for url in overpass_urls:
    try:
        print(f"Consultando Overpass API en: {url} ...")
        response = requests.get(url, params={'data': overpass_query}, timeout=100)
        response.raise_for_status()
        data = response.json()
        print("Consulta exitosa.")
        break 
    except Exception as e:
        print(f"Fallo al consultar {url}: {e}")
        time.sleep(2) 

m = folium.Map(location=[40.69, -73.97], zoom_start=12)
comisarias_data = []

if data and 'elements' in data:
    elements = data['elements']
    print(f"Se encontraron {len(elements)} elementos.")
    
    for element in elements:
        lat = element.get('lat') or element.get('center', {}).get('lat')
        lon = element.get('lon') or element.get('center', {}).get('lon')
        tags = element.get('tags', {})
        name = tags.get('name', 'Comisaría desconocida')
        
        if lat and lon:
            folium.Marker(
                [lat, lon], 
                popup=name, 
                icon=folium.Icon(color='blue', icon='info-sign')
            ).add_to(m)
            
            record = {
                'id': element.get('id'),
                'type': element.get('type'),
                'latitude': lat,
                'longitude': lon,
                'name': name,
                'amenity': tags.get('amenity'),
                'operator': tags.get('operator'),
                'operator_type': tags.get('operator:type'),
                'police': tags.get('police'),
                'addr_street': tags.get('addr:street'),
                'addr_housenumber': tags.get('addr:housenumber'),
                'addr_postcode': tags.get('addr:postcode'),
                'addr_city': tags.get('addr:city'),
                'phone': tags.get('phone'),
                'website': tags.get('website'),
                'opening_hours': tags.get('opening_hours'),
                'wheelchair': tags.get('wheelchair'),
                'all_tags': str(tags)
            }
            comisarias_data.append(record)
else:
    print("No se pudieron obtener datos de ninguna API o la respuesta estaba vacía.")

output_file = "mapa_osm.html"
m.save(output_file)
print(f"Mapa guardado exitosamente en: {os.path.abspath(output_file)}")

if comisarias_data:
    df = pd.DataFrame(comisarias_data)
    csv_file = "comisarias_policia_madrid.csv"
    df.to_csv(csv_file, index=False, encoding='utf-8-sig')
    print(f"Datos guardados exitosamente en CSV: {os.path.abspath(csv_file)}")
    print(f"\nResumen de datos guardados:")
    print(f"- Total de registros: {len(df)}")
    print(f"- Columnas: {', '.join(df.columns)}")
    print(f"\nPrimeras filas del dataset:")
    print(df.head())
else:
    print("No hay datos para guardar en CSV.")


Consultando Overpass API en: https://overpass-api.de/api/interpreter ...
Fallo al consultar https://overpass-api.de/api/interpreter: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter?data=%0A%5Bout%3Ajson%5D%5Btimeout%3A90%5D%3B%0A%28%0A++node%5B%22amenity%22%3D%22police%22%5D%2840.49%2C-74.26%2C40.92%2C-73.69%29%3B%0A++way%5B%22amenity%22%3D%22police%22%5D%2840.49%2C-74.26%2C40.92%2C-73.69%29%3B%0A++relation%5B%22amenity%22%3D%22police%22%5D%2840.49%2C-74.26%2C40.92%2C-73.69%29%3B%0A%29%3B%0Aout+center%3B%0A
Consultando Overpass API en: https://maps.mail.ru/osm/tools/overpass/api/interpreter ...
Consulta exitosa.
Se encontraron 214 elementos.
Mapa guardado exitosamente en: c:\Users\capec\OneDrive\Escritorio\Tesis Repo\mapa_osm.html
Datos guardados exitosamente en CSV: c:\Users\capec\OneDrive\Escritorio\Tesis Repo\comisarias_policia_madrid.csv

Resumen de datos guardados:
- Total de registros: 214
- Columnas: id, type, latitude, longitude, name, amenity

## 3. Procesamiento y Limpieza de Datos de Delitos NYPD

Limpieza y corrección del archivo CSV de datos de delitos de la NYPD:
- Conversión de coordenadas de formato europeo (coma) a formato estándar (punto)
- Limpieza de valores "(null)" y valores vacíos
- Identificación y manejo de coordenadas inválidas
- Exportación a CSV limpio y correcto

In [8]:
import pandas as pd
import numpy as np
import os

input_file = "NYPD_Complaint_Data_Current_(Year_To_Date)_20260108.csv"
output_file = "NYPD_Complaint_Data_Cleaned.csv"

print(f"Leyendo archivo: {input_file}")
print("Este proceso puede tardar varios minutos debido al tamaño del archivo...\n")

try:
    df = pd.read_csv(input_file, low_memory=False, encoding='utf-8')
    print(f"✓ Archivo cargado: {len(df):,} registros, {len(df.columns)} columnas")
    print(f"Columnas: {', '.join(df.columns[:10])}... (y {len(df.columns)-10} más)\n")
except Exception as e:
    print(f"Error al leer el archivo: {e}")
    print("Intentando con diferentes codificaciones...")
    try:
        df = pd.read_csv(input_file, low_memory=False, encoding='latin-1')
        print(f"✓ Archivo cargado con encoding latin-1")
    except Exception as e2:
        print(f"Error: {e2}")
        raise

print("=== Limpieza de Datos ===\n")

print("1. Limpiando valores '(null)' y convirtiendo a NaN...")
df = df.replace('(null)', np.nan)
df = df.replace('(null)', np.nan)
print(f"   ✓ Valores '(null)' reemplazados\n")

print("2. Procesando coordenadas (Latitude y Longitude)...")
if 'Latitude' in df.columns and 'Longitude' in df.columns:
    def convert_coordinate(coord):
        if pd.isna(coord):
            return np.nan
        coord_str = str(coord).strip()
        if coord_str == '' or coord_str == '0' or coord_str == '0.0':
            return np.nan
        coord_str = coord_str.replace(',', '.')
        try:
            coord_float = float(coord_str)
            if coord_float == 0.0:
                return np.nan
            return coord_float
        except:
            return np.nan
    
    df['Latitude'] = df['Latitude'].apply(convert_coordinate)
    df['Longitude'] = df['Longitude'].apply(convert_coordinate)
    
    valid_coords = df.dropna(subset=['Latitude', 'Longitude'])
    invalid_coords = len(df) - len(valid_coords)
    
    print(f"   ✓ Coordenadas convertidas de formato europeo (coma) a estándar (punto)")
    print(f"   ✓ Registros con coordenadas válidas: {len(valid_coords):,}")
    print(f"   ✓ Registros con coordenadas inválidas/faltantes: {invalid_coords:,}\n")
else:
    print("   ⚠ Columnas 'Latitude' o 'Longitude' no encontradas\n")

print("3. Limpiando columnas de texto...")
text_columns = df.select_dtypes(include=['object']).columns
for col in text_columns:
    df[col] = df[col].replace('(null)', np.nan)
    df[col] = df[col].replace('', np.nan)
print(f"   ✓ {len(text_columns)} columnas de texto limpiadas\n")

print("4. Creando columna de validación de coordenadas...")
df['has_valid_coords'] = df['Latitude'].notna() & df['Longitude'].notna() & (df['Latitude'] != 0) & (df['Longitude'] != 0)
print(f"   ✓ Columna 'has_valid_coords' creada\n")

print("5. Guardando archivo CSV limpio...")
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"   ✓ Archivo guardado: {os.path.abspath(output_file)}\n")

print("=== Resumen Final ===\n")
print(f"Total de registros: {len(df):,}")
print(f"Total de columnas: {len(df.columns)}")
print(f"Registros con coordenadas válidas: {df['has_valid_coords'].sum():,} ({df['has_valid_coords'].sum()/len(df)*100:.2f}%)")
print(f"Registros sin coordenadas: {(~df['has_valid_coords']).sum():,} ({(~df['has_valid_coords']).sum()/len(df)*100:.2f}%)\n")

if 'OFNS_DESC' in df.columns:
    print("Top 10 tipos de delitos:")
    print(df['OFNS_DESC'].value_counts().head(10))
    print()

if 'BORO_NM' in df.columns:
    print("Distribución por borough:")
    print(df['BORO_NM'].value_counts())
    print()

print("Primeras filas del dataset limpio:")
print(df[['CMPLNT_NUM', 'BORO_NM', 'OFNS_DESC', 'Latitude', 'Longitude', 'has_valid_coords']].head(10))

Leyendo archivo: NYPD_Complaint_Data_Current_(Year_To_Date)_20260108.csv
Este proceso puede tardar varios minutos debido al tamaño del archivo...

✓ Archivo cargado: 438,556 registros, 36 columnas
Columnas: CMPLNT_NUM, ADDR_PCT_CD, BORO_NM, CMPLNT_FR_DT, CMPLNT_FR_TM, CMPLNT_TO_DT, CMPLNT_TO_TM, CRM_ATPT_CPTD_CD, HADEVELOPT, HOUSING_PSA... (y 26 más)

=== Limpieza de Datos ===

1. Limpiando valores '(null)' y convirtiendo a NaN...
   ✓ Valores '(null)' reemplazados

2. Procesando coordenadas (Latitude y Longitude)...
   ✓ Coordenadas convertidas de formato europeo (coma) a estándar (punto)
   ✓ Registros con coordenadas válidas: 435,650
   ✓ Registros con coordenadas inválidas/faltantes: 2,906

3. Limpiando columnas de texto...
   ✓ 26 columnas de texto limpiadas

4. Creando columna de validación de coordenadas...
   ✓ Columna 'has_valid_coords' creada

5. Guardando archivo CSV limpio...
   ✓ Archivo guardado: c:\Users\capec\OneDrive\Escritorio\Tesis Repo\NYPD_Complaint_Data_Cleaned.cs

## 4. Visualización de Delitos en Mapa

Creación de mapa interactivo con los datos de delitos procesados, mostrando la ubicación geográfica de cada incidente.

In [9]:
import folium
import pandas as pd
import os
from folium.plugins import MarkerCluster

cleaned_file = "NYPD_Complaint_Data_Cleaned.csv"

if not os.path.exists(cleaned_file):
    print(f"Error: El archivo {cleaned_file} no existe.")
    print("Por favor, ejecuta primero la celda de limpieza de datos.")
else:
    print(f"Cargando datos limpios desde: {cleaned_file}")
    df = pd.read_csv(cleaned_file, low_memory=False)
    
    df_valid = df[df['has_valid_coords'] == True].copy()
    print(f"Total de registros con coordenadas válidas: {len(df_valid):,}\n")
    
    if len(df_valid) == 0:
        print("No hay registros con coordenadas válidas para mapear.")
    else:
        print("Creando mapa interactivo...")
        print("NOTA: Para archivos grandes, se recomienda filtrar los datos primero.\n")
        
        sample_size = min(10000, len(df_valid))
        if len(df_valid) > 10000:
            print(f"⚠ Limitando a {sample_size:,} registros para mejor rendimiento.")
            print("   Puedes filtrar los datos antes de crear el mapa.\n")
            df_map = df_valid.sample(n=sample_size, random_state=42)
        else:
            df_map = df_valid
        
        m = folium.Map(location=[40.7128, -73.9352], zoom_start=11)
        
        marker_cluster = MarkerCluster().add_to(m)
        
        tipo_colores = {
            'FELONY': 'red',
            'MISDEMEANOR': 'orange',
            'VIOLATION': 'yellow',
            'OTHER': 'gray'
        }
        
        print(f"Mapeando {len(df_map):,} delitos...")
        for idx, row in df_map.iterrows():
            lat = row['Latitude']
            lon = row['Longitude']
            
            if pd.notna(lat) and pd.notna(lon):
                ofns_desc = str(row.get('OFNS_DESC', 'Unknown'))
                law_cat = str(row.get('LAW_CAT_CD', 'OTHER'))
                fecha = str(row.get('CMPLNT_FR_DT', 'Unknown'))
                boro = str(row.get('BORO_NM', 'Unknown'))
                prem_type = str(row.get('PREM_TYP_DESC', 'Unknown'))
                
                color = tipo_colores.get(law_cat, 'gray')
                
                popup_html = f"""
                <b>Tipo de Delito:</b> {ofns_desc}<br>
                <b>Categoría:</b> {law_cat}<br>
                <b>Fecha:</b> {fecha}<br>
                <b>Borough:</b> {boro}<br>
                <b>Lugar:</b> {prem_type}<br>
                <b>Coordenadas:</b> {lat:.6f}, {lon:.6f}
                """
                
                folium.Marker(
                    [lat, lon],
                    popup=folium.Popup(popup_html, max_width=300),
                    icon=folium.Icon(color=color, icon='exclamation-triangle', prefix='fa'),
                    tooltip=f"{ofns_desc[:30]}... - {fecha}"
                ).add_to(marker_cluster)
        
        output_file = "mapa_delitos_nypd.html"
        m.save(output_file)
        print(f"\n✓ Mapa guardado en: {os.path.abspath(output_file)}")
        print(f"✓ Total de delitos mapeados: {len(df_map):,}")
        
        print(f"\n=== Estadísticas ===")
        if 'LAW_CAT_CD' in df_map.columns:
            print("\nDistribución por categoría legal:")
            print(df_map['LAW_CAT_CD'].value_counts())
        
        if 'BORO_NM' in df_map.columns:
            print("\nDistribución por borough:")
            print(df_map['BORO_NM'].value_counts())
        
        print(f"\nPara ver el mapa completo, abre el archivo: {output_file}")

Cargando datos limpios desde: NYPD_Complaint_Data_Cleaned.csv
Total de registros con coordenadas válidas: 435,650

Creando mapa interactivo...
NOTA: Para archivos grandes, se recomienda filtrar los datos primero.

⚠ Limitando a 10,000 registros para mejor rendimiento.
   Puedes filtrar los datos antes de crear el mapa.

Mapeando 10,000 delitos...


C:\Users\capec\AppData\Local\Temp\ipykernel_7124\3882677196.py:69: UserWarning: color argument of Icon should be one of: {'gray', 'lightblue', 'darkblue', 'beige', 'red', 'blue', 'white', 'pink', 'darkred', 'black', 'lightgreen', 'cadetblue', 'darkpurple', 'green', 'darkgreen', 'orange', 'purple', 'lightred', 'lightgray'}.
  icon=folium.Icon(color=color, icon='exclamation-triangle', prefix='fa'),



✓ Mapa guardado en: c:\Users\capec\OneDrive\Escritorio\Tesis Repo\mapa_delitos_nypd.html
✓ Total de delitos mapeados: 10,000

=== Estadísticas ===

Distribución por categoría legal:
LAW_CAT_CD
MISDEMEANOR    5221
FELONY         3202
VIOLATION      1577
Name: count, dtype: int64

Distribución por borough:
BORO_NM
BROOKLYN         2788
MANHATTAN        2359
BRONX            2280
QUEENS           2143
STATEN ISLAND     400
Name: count, dtype: int64

Para ver el mapa completo, abre el archivo: mapa_delitos_nypd.html


## 5. Mapa Combinado: Delitos y Comisarías

Visualización combinada de delitos y comisarías de policía en un mismo mapa interactivo con capas separadas para poder activar/desactivar cada tipo de marcador.

In [18]:
import folium
import pandas as pd
import os
from folium.plugins import MarkerCluster

print("=== Creando Mapa Combinado: Delitos y Comisarías ===\n")

delitos_file = "NYPD_Complaint_Data_Cleaned.csv"
comisarias_file = "comisarias_policia_madrid.csv"

m = folium.Map(location=[40.7128, -73.9352], zoom_start=11)

fg_delitos = folium.FeatureGroup(name='Delitos', show=True)
fg_comisarias = folium.FeatureGroup(name='Comisarías de Policía', show=True)

print("1. Cargando datos de delitos...")
if os.path.exists(delitos_file):
    df_delitos = pd.read_csv(delitos_file, low_memory=False)
    df_delitos_valid = df_delitos[df_delitos['has_valid_coords'] == True].copy()
    
    sample_size = min(5000, len(df_delitos_valid))
    if len(df_delitos_valid) > sample_size:
        df_delitos_map = df_delitos_valid.sample(n=sample_size, random_state=42)
        print(f"   ✓ {len(df_delitos_map):,} delitos seleccionados (muestra de {len(df_delitos_valid):,} totales)")
    else:
        df_delitos_map = df_delitos_valid
        print(f"   ✓ {len(df_delitos_map):,} delitos cargados")
    
    tipo_colores = {
        'FELONY': 'red',
        'MISDEMEANOR': 'orange',
        'VIOLATION': 'yellow'
    }
    
    marker_cluster_delitos = MarkerCluster().add_to(fg_delitos)
    
    print("   Mapeando delitos...")
    for idx, row in df_delitos_map.iterrows():
        lat = row['Latitude']
        lon = row['Longitude']
        
        if pd.notna(lat) and pd.notna(lon):
            ofns_desc = str(row.get('OFNS_DESC', 'Unknown'))
            law_cat = str(row.get('LAW_CAT_CD', 'OTHER'))
            fecha = str(row.get('CMPLNT_FR_DT', 'Unknown'))
            boro = str(row.get('BORO_NM', 'Unknown'))
            
            color = tipo_colores.get(law_cat, 'gray')
            
            popup_html = f"""
            <div style="width:250px">
            <b>DELITO</b><br>
            <b>Tipo:</b> {ofns_desc}<br>
            <b>Categoría:</b> {law_cat}<br>
            <b>Fecha:</b> {fecha}<br>
            <b>Borough:</b> {boro}<br>
            <b>Coordenadas:</b> {lat:.6f}, {lon:.6f}
            </div>
            """
            
            folium.Marker(
                [lat, lon],
                popup=folium.Popup(popup_html, max_width=300),
                icon=folium.Icon(color=color, icon='exclamation-triangle', prefix='fa'),
                tooltip=f"Delito: {ofns_desc[:25]}..."
            ).add_to(marker_cluster_delitos)
    
    fg_delitos.add_to(m)
    print("   ✓ Capa de delitos agregada\n")
else:
    print(f"   ⚠ Archivo {delitos_file} no encontrado\n")

print("2. Cargando datos de comisarías...")
if os.path.exists(comisarias_file):
    df_comisarias = pd.read_csv(comisarias_file, encoding='utf-8-sig')
    print(f"   Total de comisarías en archivo: {len(df_comisarias)}")
    
    df_comisarias_valid = df_comisarias[
        (df_comisarias['latitude'].notna()) & 
        (df_comisarias['longitude'].notna()) &
        (df_comisarias['latitude'] != 0) &
        (df_comisarias['longitude'] != 0)
    ].copy()
    
    print(f"   ✓ {len(df_comisarias_valid)} comisarías con coordenadas válidas")
    
    marker_cluster_comisarias = MarkerCluster().add_to(fg_comisarias)
    
    comisarias_mapeadas = 0
    print("   Mapeando comisarías...")
    for idx, row in df_comisarias_valid.iterrows():
        try:
            lat = float(row['latitude'])
            lon = float(row['longitude'])
            
            if pd.notna(lat) and pd.notna(lon) and lat != 0 and lon != 0:
                name = str(row.get('name', 'Comisaría desconocida'))
                operator = str(row.get('operator', 'N/A'))
                addr = str(row.get('addr_street', 'N/A'))
                
                popup_html = f"""
                <div style="width:250px">
                <b>COMISARIA</b><br>
                <b>Nombre:</b> {name}<br>
                <b>Operador:</b> {operator}<br>
                <b>Direccion:</b> {addr}<br>
                <b>Coordenadas:</b> {lat:.6f}, {lon:.6f}
                </div>
                """
                
                folium.Marker(
                    [lat, lon],
                    popup=folium.Popup(popup_html, max_width=300),
                    icon=folium.Icon(color='blue', icon='shield', prefix='fa'),
                    tooltip=f"Comisaría: {name[:30]}..."
                ).add_to(marker_cluster_comisarias)
                comisarias_mapeadas += 1
        except Exception as e:
            continue
    
    fg_comisarias.add_to(m)
    print(f"   ✓ {comisarias_mapeadas} comisarías mapeadas exitosamente")
    print("   ✓ Capa de comisarías agregada\n")
else:
    print(f"   ⚠ Archivo {comisarias_file} no encontrado")
    print("   Ejecuta primero la celda que obtiene las comisarías desde OSM\n")

folium.LayerControl(collapsed=False).add_to(m)

output_file = "mapa_combinado_delitos_comisarias.html"
m.save(output_file)
print(f"✓ Mapa combinado guardado en: {os.path.abspath(output_file)}")
print(f"\n=== Resumen ===")
if os.path.exists(delitos_file) and os.path.exists(comisarias_file):
    print(f"Delitos mapeados: {len(df_delitos_map):,}")
    if 'comisarias_mapeadas' in locals():
        print(f"Comisarías mapeadas: {comisarias_mapeadas}")
    else:
        print(f"Comisarías mapeadas: {len(df_comisarias_valid) if 'df_comisarias_valid' in locals() else 0}")
    print(f"\nEl mapa incluye controles de capas en la esquina superior derecha")
    print(f"para activar/desactivar cada tipo de marcador.")
    print(f"\nNOTA: Si las comisarías aparecen agrupadas, haz zoom in para verlas individualmente.")
else:
    print("Verifica que ambos archivos existan antes de crear el mapa.")

=== Creando Mapa Combinado: Delitos y Comisarías ===

1. Cargando datos de delitos...
   ✓ 5,000 delitos seleccionados (muestra de 435,650 totales)
   Mapeando delitos...


C:\Users\capec\AppData\Local\Temp\ipykernel_7124\2629014207.py:64: UserWarning: color argument of Icon should be one of: {'gray', 'lightblue', 'darkblue', 'beige', 'red', 'blue', 'white', 'pink', 'darkred', 'black', 'lightgreen', 'cadetblue', 'darkpurple', 'green', 'darkgreen', 'orange', 'purple', 'lightred', 'lightgray'}.
  icon=folium.Icon(color=color, icon='exclamation-triangle', prefix='fa'),


   ✓ Capa de delitos agregada

2. Cargando datos de comisarías...
   Total de comisarías en archivo: 214
   ✓ 214 comisarías con coordenadas válidas
   Mapeando comisarías...
   ✓ 214 comisarías mapeadas exitosamente
   ✓ Capa de comisarías agregada

✓ Mapa combinado guardado en: c:\Users\capec\OneDrive\Escritorio\Tesis Repo\mapa_combinado_delitos_comisarias.html

=== Resumen ===
Delitos mapeados: 5,000
Comisarías mapeadas: 214

El mapa incluye controles de capas en la esquina superior derecha
para activar/desactivar cada tipo de marcador.

NOTA: Si las comisarías aparecen agrupadas, haz zoom in para verlas individualmente.
